# Lia — tentativa de LoRA Qwen3-TTS no Colab T4

Notebook **separado** do benchmark e do notebook de fine-tuning completo. Usa o ZIP curado que você já baixou; não pede nem retranscreve o WAV original. A saída é um adapter LoRA pequeno. Dados, checkpoints e áudios gerados ficam no runtime privado Colab.

**Tentativa experimental para T4, sem garantia de caber:** LoRA reduz pesos treináveis/estado do otimizador, mas ainda precisa guardar ativações. Primeiro executa exatamente um passo de treino em FP32 + gradient checkpointing + SDPA. Só inicie a época completa depois de o smoke test passar. Se houver OOM, o subprocesso termina antes do treino longo.

A receita comunitária `instavar/qwen3-tts-lora-finetuning` está pinada e aplicada a um commit Qwen upstream específico, com correções conhecidas de alinhamento de loss e `text_projection`. Não é o trainer oficial nem valida qualidade PT-BR. Os rótulos emotion do ZIP são metadados: o trainer não os condiciona diretamente; a adaptação usa os áudios expressivos e a inferência pede emoção por `instruct`.

**Configuração inicial:** Qwen 0.6B, base FP32 (não BF16/FP16), SDPA, batch 1, gradient accumulation 4, rank 4, Q/K/V/O, LR 2e-6, uma época. Os pesos-base não são alterados.

In [ ]:
# 0. Dependências auxiliares isoladas; mantém Torch/Torchaudio CUDA do Colab
from pathlib import Path
import os, sys, subprocess, json, csv, shutil, zipfile, hashlib, importlib.util, gc
import torch
ROOT=Path('/content/lia_qwen_lora'); ROOT.mkdir(parents=True,exist_ok=True)
DEPS=ROOT/'python_deps'; DEPS.mkdir(exist_ok=True)
packages=['transformers==4.57.3','accelerate==1.12.0','peft==0.17.1','huggingface_hub==0.36.2','safetensors==0.7.0','tokenizers==0.22.1','librosa==0.11.0','soxr==0.5.0.post1','einops==0.8.1','onnxruntime==1.23.2']
marker=DEPS/'.lora_deps_ready'
if not marker.exists():
    subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check','--no-deps','--target',str(DEPS),*packages],check=True)
    marker.write_text('\n'.join(packages)+'\n',encoding='utf-8')
sys.path.insert(0,str(DEPS))
# Qwen é áudio; impede auto-detecção de uma torchvision incompatível não usada.
import transformers.utils.import_utils as _hf_import_utils
_hf_import_utils._torchvision_available=False
import transformers, accelerate, peft, huggingface_hub, soundfile as sf
from google.colab import files
from IPython.display import display, Audio
if not torch.cuda.is_available(): raise RuntimeError('Selecione uma GPU CUDA (T4) no runtime Colab.')
props=torch.cuda.get_device_properties(0)
print(f'GPU: {props.name} | VRAM: {props.total_memory/1024**3:.1f} GiB | BF16 reportado: {torch.cuda.is_bf16_supported()}')
print('Torch ativo:',torch.__version__,'| Transformers isolado:',transformers.__version__,'| PEFT:',peft.__version__)
print('Dados/checkpoints em',ROOT)
print('Selecione o ZIP curado já baixado.')
uploaded=files.upload()
if len(uploaded)!=1: raise ValueError('Envie exatamente um ZIP.')
zip_name,zip_bytes=next(iter(uploaded.items()))
if not zip_name.lower().endswith('.zip'): raise ValueError('Esperado um arquivo ZIP.')
bundle=ROOT/Path(zip_name).name; bundle.write_bytes(zip_bytes); del zip_bytes,uploaded; gc.collect()
with zipfile.ZipFile(bundle) as zf:
    root_resolved=ROOT.resolve()
    for item in zf.infolist():
        target=(ROOT/item.filename).resolve()
        if target!=root_resolved and root_resolved not in target.parents: raise ValueError('Caminho inseguro no ZIP; extração bloqueada.')
    zf.extractall(ROOT)
manifest=json.loads((ROOT/'training_bundle_manifest.json').read_text(encoding='utf-8'))
if manifest.get('format')!='lia-qwen-training-bundle-v1': raise ValueError('Formato de pacote inesperado.')
for required in ('train_raw.jsonl','holdout_raw.jsonl','review.csv','review_approval.json','emotion_audit.csv','speaker_reference.wav'):
    if not (ROOT/required).is_file(): raise FileNotFoundError(f'Arquivo obrigatório ausente: {required}')
review_hash=hashlib.sha256((ROOT/'review.csv').read_bytes()).hexdigest()
approval=json.loads((ROOT/'review_approval.json').read_text(encoding='utf-8'))
if approval.get('review_csv_sha256')!=review_hash or manifest.get('review_csv_sha256')!=review_hash: raise RuntimeError('Hash da revisão não corresponde à aprovação; pacote alterado/incompleto.')
print('Pacote curado:',manifest.get('train_rows'),'treino;',manifest.get('holdout_rows'),'holdout; review hash',review_hash[:12])

## 1. Validar e limitar duração para a T4
Expande caminhos portáteis/antigos do ZIP para caminhos absolutos locais, verifica os WAVs e limita a 12 s para reduzir memória de ativações. Os originais não são alterados; holdout permanece separado.

In [ ]:
# 1. Paths locais + teto de duração (apenas para esta experiência T4)
from pathlib import Path
import soundfile as sf
MAX_CLIP_SECONDS=12.0

def resolve_bundled_audio(value):
    p=Path(value)
    if p.is_absolute() and p.is_file() and p.is_relative_to(ROOT.resolve()): return p.resolve()
    # O JSONL exportado pode conter o caminho absoluto do runtime antigo; no ZIP, o áudio é portátil.
    matches=list(ROOT.rglob(p.name))
    matches=[x.resolve() for x in matches if x.is_file() and x.suffix.lower()=='.wav']
    if len(matches)!=1: raise FileNotFoundError(f'Não consegui resolver univocamente o áudio {value!r}: {matches[:4]}')
    return matches[0]

def prepare_split(src_name,dst_name):
    kept=[]; dropped=[]; refs=set()
    for line_no,line in enumerate((ROOT/src_name).read_text(encoding='utf-8').splitlines(),1):
        if not line.strip(): continue
        row=json.loads(line); audio=resolve_bundled_audio(row['audio']); ref=resolve_bundled_audio(row['ref_audio'])
        if not audio.is_relative_to(ROOT.resolve()) or not ref.is_relative_to(ROOT.resolve()): raise ValueError('Áudio fora do ZIP/runtime privado.')
        info=sf.info(audio)
        if info.channels!=1 or info.samplerate!=24000: raise ValueError(f'{audio.name}: esperava mono 24 kHz, recebi {info.channels}ch/{info.samplerate}Hz')
        refs.add(ref)
        if info.duration>MAX_CLIP_SECONDS: dropped.append((audio.name,info.duration)); continue
        row['audio']=str(audio); row['ref_audio']=str(ref); kept.append(row)
    if len(refs)!=1: raise ValueError(f'{src_name}: esperado um ref_audio fixo, encontrei {len(refs)}.')
    if len(kept)<8: raise ValueError(f'{src_name}: restaram só {len(kept)} recortes <= {MAX_CLIP_SECONDS}s.')
    (ROOT/dst_name).write_text(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in kept),encoding='utf-8')
    return kept,dropped
train_rows,dropped_train=prepare_split('train_raw.jsonl','train_t4.jsonl')
val_rows,dropped_val=prepare_split('holdout_raw.jsonl','holdout_t4.jsonl')
print(f'Treino: {len(train_rows)} recortes, {sum(sf.info(r["audio"]).duration for r in train_rows)/60:.1f} min; holdout: {len(val_rows)}.')
print(f'Excluídos temporariamente por >{MAX_CLIP_SECONDS}s: treino {len(dropped_train)}, holdout {len(dropped_val)}.')
print('Emoções anotadas (metadado, não condicionamento direto):',sorted({r.get('emotion_audit','nao_rotulado') for r in train_rows}))

## 2. Pin Qwen + correções LoRA e preparar modelo
O patch comunitário se aplica ao commit upstream fixo `0c6a7c…`; o notebook verifica a revisão e o `git apply --check`. Ajustes estritos para T4: FP32, sem TensorBoard, gradient checkpointing, batch do tokenizer 1 e SDPA. Se o código upstream divergir, a célula para em vez de fazer patch aproximado.

In [ ]:
# 2. Repositórios e snapshots de modelo pinados
QWEN_COMMIT='0c6a7cbb6c8421a46332f8c2434c7825c4c855ef'
COMPANION_COMMIT='079db4ba844037f970b488dc96b32e5c7490642f'
MODEL_ID='Qwen/Qwen3-TTS-12Hz-0.6B-Base'; MODEL_REV='5d83992436eae1d760afd27aff78a71d676296fc'
TOKENIZER_ID='Qwen/Qwen3-TTS-Tokenizer-12Hz'; TOKENIZER_REV='7dd38ad4e9bad454aae9cd937d0cd577604fe229'
QWEN_DIR=ROOT/'Qwen3-TTS'; COMP_DIR=ROOT/'qwen3-tts-lora-finetuning'
def clone_pinned(url,dest,sha):
    if not dest.exists(): subprocess.run(['git','clone','--quiet','--no-checkout',url,str(dest)],check=True)
    subprocess.run(['git','-C',str(dest),'fetch','--quiet','--depth','1','origin',sha],check=True)
    subprocess.run(['git','-C',str(dest),'checkout','--quiet','--force',sha],check=True)
    got=subprocess.check_output(['git','-C',str(dest),'rev-parse','HEAD'],text=True).strip()
    if got!=sha: raise RuntimeError(f'Commit inesperado: {got}')
clone_pinned('https://github.com/QwenLM/Qwen3-TTS.git',QWEN_DIR,QWEN_COMMIT)
clone_pinned('https://github.com/instavar/qwen3-tts-lora-finetuning.git',COMP_DIR,COMPANION_COMMIT)
patch=COMP_DIR/'patches'/'0001-qwen3-tts-lora.patch'
subprocess.run(['git','-C',str(QWEN_DIR),'apply','--check',str(patch)],check=True)
subprocess.run(['git','-C',str(QWEN_DIR),'apply',str(patch)],check=True)
trainer=QWEN_DIR/'finetuning'/'sft_12hz_lora.py'
source=trainer.read_text(encoding='utf-8')
def replace_once(text,old,new,label):
    n=text.count(old)
    if n!=1: raise RuntimeError(f'Patch T4 abortado: {label} devia ocorrer uma vez, ocorreu {n}.')
    return text.replace(old,new,1)
source=replace_once(source,'torch_dtype=torch.bfloat16,','torch_dtype=torch.float32,','FP32 para T4')
source=replace_once(source,'log_with="tensorboard",','log_with=None,','sem TensorBoard')
source=replace_once(source,'    config = AutoConfig.from_pretrained(MODEL_PATH)\n','''    config = AutoConfig.from_pretrained(MODEL_PATH)
    qwen3tts.model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    qwen3tts.model.config.use_cache = False
''','gradient checkpointing')
source=replace_once(source,'    parser.add_argument("--eval_every", type=int, default=1)\n','''    parser.add_argument("--eval_every", type=int, default=1)
    parser.add_argument("--max_train_steps", type=int, default=0)
''','limite do smoke')
source=replace_once(source,'            if step % 10 == 0:\n                accelerator.print(f"Epoch {epoch} | Step {step} | Loss: {loss.item():.4f}")\n','''            if step % 10 == 0:
                accelerator.print(f"Epoch {epoch} | Step {step} | Loss: {loss.item():.4f}")
            if args.max_train_steps and step + 1 >= args.max_train_steps:
                peak=torch.cuda.max_memory_allocated()/1024**3 if torch.cuda.is_available() else 0.0
                accelerator.print(f"Smoke test encerrou em {step+1} passo; pico CUDA alocado {peak:.2f} GiB; nenhum checkpoint salvo.")
                return
''','saída limitada do smoke')
trainer.write_text(source,encoding='utf-8')
prep=QWEN_DIR/'finetuning'/'prepare_data.py'; ps=prep.read_text(encoding='utf-8')
if ps.count('BATCH_INFER_NUM = 32')!=1: raise RuntimeError('Batch do tokenizer mudou; revisão necessária.')
prep.write_text(ps.replace('BATCH_INFER_NUM = 32','BATCH_INFER_NUM = 1',1),encoding='utf-8')
infer=QWEN_DIR/'finetuning'/'infer_lora_custom_voice.py'; ins=infer.read_text(encoding='utf-8')
ins=replace_once(ins,'    parser.add_argument("--language", default="auto")\n','''    parser.add_argument("--language", default="auto")
    parser.add_argument("--instruct", default="")
''','instruct CLI')
ins=replace_once(ins,'        language=args.language,\n        max_new_tokens=args.max_new_tokens,\n','''        language=args.language,
        instruct=args.instruct or None,
        max_new_tokens=args.max_new_tokens,
''','passagem do instruct')
infer.write_text(ins,encoding='utf-8')
os.environ['HF_HOME']=str(ROOT/'hf_home')
from huggingface_hub import snapshot_download
model_path=snapshot_download(MODEL_ID,revision=MODEL_REV)
tokenizer_path=snapshot_download(TOKENIZER_ID,revision=TOKENIZER_REV)
train_env=os.environ.copy(); train_env['HF_HOME']=str(ROOT/'hf_home')
(DEPS/'sitecustomize.py').write_text('try:\n import transformers.utils.import_utils as _u\n _u._torchvision_available=False\nexcept Exception:\n pass\n',encoding='utf-8')
train_env['PYTHONPATH']=os.pathsep.join([str(DEPS),str(QWEN_DIR),train_env.get('PYTHONPATH','')])
train_env['PYTHONUNBUFFERED']='1'
print('Patch fixo aplicado. Modelos baixados; trainer: FP32, SDPA, rank 4, batch 1, gradient checkpointing.')

## 3. Tokenizar os códigos de áudio
Executa o tokenizer oficial sobre treino e holdout. Batch 1 para limitar a VRAM. Ainda não treina.

In [ ]:
# 3. Preparar códigos de áudio
prep=QWEN_DIR/'finetuning'/'prepare_data.py'
for src,dst in [('train_t4.jsonl','train_t4_codes.jsonl'),('holdout_t4.jsonl','holdout_t4_codes.jsonl')]:
    subprocess.run([sys.executable,'-u',str(prep),'--device','cuda:0','--tokenizer_model_path',tokenizer_path,
                    '--input_jsonl',str(ROOT/src),'--output_jsonl',str(ROOT/dst)],check=True,env=train_env)
    if not (ROOT/dst).is_file() or (ROOT/dst).stat().st_size==0: raise RuntimeError(f'Preparação incompleta: {dst}')
print('JSONLs codificados localmente; nenhum áudio enviado a serviço.')

## 4. Smoke test: exatamente um passo
Carrega o modelo, executa exatamente um forward/backward e um update real do adapter (acumulação 1), depois encerra sem salvar checkpoint. Se OOM/incompatibilidade, a célula levanta erro e o treino longo não começa.

In [ ]:
# 4. Um passo descartável
BASE_ARGS=[sys.executable,'-u',str(trainer),'--init_model_path',model_path,'--output_model_path',str(ROOT/'smoke_discarded'),
 '--train_jsonl',str(ROOT/'train_t4_codes.jsonl'),'--batch_size','1','--lr','2e-6','--num_epochs','1','--speaker_name','lia_lora',
 '--gradient_accumulation_steps','1','--mixed_precision','no','--attn_implementation','sdpa','--lora_rank','4','--lora_alpha','8',
 '--lora_dropout','0.05','--lora_target_modules','q_proj,k_proj,v_proj,o_proj','--max_train_steps','1']
print('Smoke test na T4; se falhar, pare aqui.',flush=True)
proc=subprocess.Popen(BASE_ARGS,env=train_env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in proc.stdout: print(line,end='',flush=True)
rc=proc.wait()
if rc: raise RuntimeError(f'Smoke test LoRA falhou (exit {rc}); treino completo NÃO iniciado.')
print('Smoke test passou. A próxima célula inicia uma época experimental.')

## 5. Treinar adapter (uma época inicial)
Experimento curto, não modelo final. Loss e holdout não avaliam expressividade nem sotaque. Adapter, configuração e embedding de speaker são salvos sem tocar na base.

In [ ]:
# 5. Executar uma época após smoke bem-sucedido
TRAIN_LORA=True
if TRAIN_LORA:
    adapter_dir=ROOT/'lora_adapter'
    cmd=[sys.executable,'-u',str(trainer),'--init_model_path',model_path,'--output_model_path',str(adapter_dir),
      '--train_jsonl',str(ROOT/'train_t4_codes.jsonl'),'--val_jsonl',str(ROOT/'holdout_t4_codes.jsonl'),
      '--batch_size','1','--lr','2e-6','--num_epochs','1','--speaker_name','lia_lora','--gradient_accumulation_steps','4',
      '--mixed_precision','no','--attn_implementation','sdpa','--lora_rank','4','--lora_alpha','8','--lora_dropout','0.05',
      '--lora_target_modules','q_proj,k_proj,v_proj,o_proj']
    proc=subprocess.Popen(cmd,env=train_env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in proc.stdout: print(line,end='',flush=True)
    rc=proc.wait()
    if rc: raise RuntimeError(f'Treino falhou (exit {rc}); veja log acima.')
    checkpoints=sorted((ROOT/'lora_adapter').glob('checkpoint-epoch-*'))
    if not checkpoints: raise FileNotFoundError('Treino terminou sem checkpoint adapter.')
    print('Adapter candidato:',checkpoints[-1])
else: print('Treino desativado.')

## 6. Comparar instrução alegre e neutra
Duas falas curtas com o mesmo texto, adapter e seed. `instruct` é comando de inferência, não rótulo supervisionado de emoção no trainer. Avalie timbre, naturalidade, PT-BR e diferença emocional.

In [ ]:
# 6. Síntese comparativa
from IPython.display import Audio,display
checkpoints=sorted((ROOT/'lora_adapter').glob('checkpoint-epoch-*'))
if not checkpoints: raise RuntimeError('Execute a célula de treino primeiro.')
adapter_path=str(checkpoints[-1]); infer=QWEN_DIR/'finetuning'/'infer_lora_custom_voice.py'
text='Você conseguiu! Eu sabia que era capaz. Estou muito orgulhosa de você!'
prompts=[('alegre','Fale em português brasileiro com alegria calorosa e entusiasmo genuíno, como quem comemora uma vitória; natural, sem gritar.'),
         ('neutra','Fale em português brasileiro de forma neutra, natural e conversacional.')]
for label,instruct in prompts:
    out=ROOT/f'lia_lora_{label}.wav'
    cmd=[sys.executable,'-u',str(infer),'--base_model_path',model_path,'--adapter_path',adapter_path,'--speaker_name','lia_lora',
      '--speaker_id','3000','--text',text,'--language','Portuguese','--output_wav',str(out),'--device','cuda:0','--dtype','fp32',
      '--attn_implementation','sdpa','--lora_scale','0.3','--max_new_tokens','128','--seed','1234','--instruct',instruct]
    subprocess.run(cmd,check=True,env=train_env)
    print(label,out); display(Audio(filename=str(out)))
print('Artefatos privados do runtime:',ROOT)

### Limites e próximos passos
- Se o smoke der OOM, a T4 pode não comportar backprop mesmo com LoRA. O ajuste conservador seguinte é teto de áudio 8 s e/ou rank 2 — não forçar precisão instável.
- Se treinar, não substitua o Qwen base automaticamente. Faça avaliação auditiva A/B; loss não mede emoção nem PT-BR.
- Técnica comunitária experimental, não full SFT oficial. Pesos públicos são baixados; ZIP, adapters e WAVs gerados permanecem no Colab. Nenhum serviço hosted recebe a gravação.